# Background Removal with BiRefNet

This notebook loads the state-of-the-art **BiRefNet** model for background removal (dichotomous image segmentation) and runs it on our test image.

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoModelForImageSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Load the Model
We load the model using `AutoModelForImageSegmentation`. Since BiRefNet uses custom hub code, we must set `trust_remote_code=True`.

In [ ]:
print("Loading BiRefNet model...")
model = AutoModelForImageSegmentation.from_pretrained("ZhengPeng7/BiRefNet", trust_remote_code=True)
model.to(device)
model.eval()
print("Model loaded successfully!")

## Process and Extract Background
We resize the image to `1024x1024` (BiRefNet standard resolution) and normalize it. After running inference, we extract the mask and apply it as the alpha channel.

In [ ]:
def extract_object(image_path):
    # 1. Load Image
    input_image = Image.open(image_path).convert("RGB")
    original_size = input_image.size

    # 2. Preprocess
    transform = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    # 3. Predict
    with torch.no_grad():
        preds = model(input_tensor)[-1].sigmoid().cpu() # Get the last output and apply sigmoid

    # 4. Post-process mask
    mask = transforms.ToPILImage()(preds[0])
    mask = mask.resize(original_size)

    # 5. Apply Mask to Original Image
    result_image = input_image.copy()
    result_image.putalpha(mask)
    return result_image, mask

print("Helper function defined.")

## Run Inference and Save Output
We run the pipeline on `test.png` and save the result as `output.png`.

In [ ]:
image_path = "test.png"
result_image, mask_image = extract_object(image_path)

# Save the transparent image
result_image.save("output.png")
print("Saved output to output.png")

# Display results
fig, ax = plt.subplots(1, 3, figsize=(18, 6))

ax[0].imshow(Image.open(image_path))
ax[0].set_title("Original")
ax[0].axis("off")

ax[1].imshow(mask_image, cmap='gray')
ax[1].set_title("Predicted Mask")
ax[1].axis("off")

ax[2].imshow(result_image)
ax[2].set_title("Final Cut-out")
ax[2].axis("off")

plt.show()